In [ ]:
# imports 
import numpy as np 
import pandas as pd 
import pingouin as pg
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
import glob
import os
import h5_utilities_module as h5u
from tqdm import tqdm as tqdm
import statsmodels.api as sm
import statsmodels.formula.api as smf
warnings.filterwarnings('ignore')
warnings.filterwarnings("ignore", message="Mean of empty slice", category=RuntimeWarning)

In [ ]:
def calculate_mean_and_interval(data, type='sem', num_samples=1000, alpha=0.05):
    """
    Calculate mean and either SEM or bootstrapped CI for each column of the input array, disregarding NaN values.
    Parameters:
    - data: 1D or 2D numpy array
    - type: str, either 'sem', 'percentile', or 'bootstrap'
    - num_samples: int, number of bootstrap samples (applicable only for type='bootstrap')
    - alpha: float, significance level for the confidence interval (applicable only for type='bootstrap')
    Returns:
    - means: scalar or 1D numpy array containing means for each column
    - interval: scalar or 1D numpy array containing SEMs or bootstrapped CIs for each column
    """
    # Handle 1D input by reshaping to 2D
    input_1d = data.ndim == 1
    if input_1d:
        data = data.reshape(-1, 1)

    nan_mask = ~np.isnan(data)
    
    nanmean_result = np.nanmean(data, axis=0)
    n_valid_values = np.sum(nan_mask, axis=0)
    
    if type == 'sem':
        nanstd_result = np.nanstd(data, axis=0)
        interval = nanstd_result / np.sqrt(n_valid_values)
        
    elif type == 'percentile':
        interval = np.mean(np.array([np.abs(nanmean_result - np.nanpercentile(data, 5, axis=0)), 
                                      np.abs(nanmean_result - np.nanpercentile(data, 95, axis=0))]))
        
    elif type == 'bootstrap':
        n_rows, n_cols = data.shape
        bootstrap_means = np.zeros((num_samples, n_cols))
        for col in range(n_cols):
            if np.sum(nan_mask[:, col]) > 0:
                bootstrap_samples = np.random.choice(data[:, col][nan_mask[:, col]], size=(num_samples, n_rows), replace=True)
                bootstrap_means[:, col] = np.nanmean(bootstrap_samples, axis=1)
            else:
                bootstrap_means[:, col] = np.nan
        ci_lower = np.percentile(bootstrap_means, 100 * (alpha / 2), axis=0)
        ci_upper = np.percentile(bootstrap_means, 100 * (1 - alpha / 2), axis=0)
        interval = np.nanmean([abs(bootstrap_means - ci_lower), abs(bootstrap_means - ci_upper)], axis=0)
        interval = np.nanmean(interval, axis=0)
    else:
        raise ValueError("Invalid 'type' argument. Use either 'sem', 'percentile', or 'bootstrap'.")
    
    # Squeeze back to scalar if input was 1D
    if input_1d:
        nanmean_result = nanmean_result.squeeze()
        interval = interval.squeeze()
    
    return nanmean_result, interval


def get_ch_and_unch_vals(bhv):
    """
    Extracts chosen (ch_val) and unchosen (unch_val) values associated with each trial.

    Parameters:
    - bhv (DataFrame): DataFrame behavioral data.

    Returns:
    - ch_val (ndarray): Array of chosen values for each trial.
    - unch_val (ndarray): Array of unchosen values for each trial. 
                          - places 0s for unchosen values on forced choice trials
    """
    ch_val = np.zeros(shape=(len(bhv, )))
    unch_val = np.zeros(shape=(len(bhv, )))

    bhv['r_val'] = bhv['r_val'].fillna(0)
    bhv['l_val'] = bhv['l_val'].fillna(0)

    ch_left = bhv['side'] == -1
    ch_right = bhv['side'] == 1

    ch_val[ch_left] = bhv['l_val'].loc[ch_left].astype(int)
    ch_val[ch_right] = bhv['r_val'].loc[ch_right].astype(int)

    unch_val[ch_left] = bhv['r_val'].loc[ch_left].astype(int)
    unch_val[ch_right] = bhv['l_val'].loc[ch_right].astype(int)

    return ch_val, unch_val


def arima_preprocess_trials(in_data, arima_params=(10, 2, 2)):
   """
   Apply ARIMA preprocessing to all trials of a time series.
   
   Parameters:
   -----------
   in_data: array [n_trials x n_times] - time series data
   arima_params: tuple - (p, d, q) ARIMA order
   
   Returns:
   --------
   whitened_data: array [n_trials x n_times] - ARIMA residuals or original data
   success_mask: array [n_trials] - boolean mask indicating ARIMA success per trial
   """
   
   n_trials, n_times = in_data.shape
   whitened_data = np.full((n_trials, n_times), np.nan)
   success_mask = np.zeros(n_trials, dtype=bool)
   
   for t in tqdm(range(n_trials)):
       try:
           model = ARIMA(in_data[t, :], order=arima_params)
           fitted = model.fit()
           whitened_data[t, :] = fitted.resid
           success_mask[t] = True
           
       except Exception:
           # Use original data if ARIMA fails
           whitened_data[t, :] = in_data[t, :]
           success_mask[t] = False
   
   return whitened_data, success_mask



def transmission_analysis(data_A, data_B, lags, time_step, win_size):
    """
    Perform sliding window transmission analysis between two neural time series.

    Tests directional predictive relationships by applying lagged regression analysis
    within sliding temporal windows. Based on the approach described in Crowe et al. 
    (2013) Nature Neuroscience for detecting information transmission between brain areas.

    Parameters
    ----------
    data_A : ndarray, shape (n_trials, n_times)
        Neural data from brain area A (e.g., posterior probabilities from decoding).
        Should be preprocessed (e.g., ARIMA whitened) to remove autocorrelation.
    data_B : ndarray, shape (n_trials, n_times)  
        Neural data from brain area B, same format as data_A.
    lags : array_like
        Lag values to test (in time bins). Should start from 0.
        e.g., np.arange(0, 9) tests lags 0-8 bins.
    time_step : float
        Time duration of each bin in milliseconds (e.g., 25.0 for 25ms bins).
    win_size : float  
        Sliding window size in milliseconds (e.g., 500.0 for 500ms windows).
        
    Returns
    -------
    a2b_f : ndarray, shape (n_lags, n_time_steps)
        F-statistics testing whether area A predicts area B at each lag and time step.
        Higher values indicate stronger predictive relationships.
    b2a_f : ndarray, shape (n_lags, n_time_steps)
        F-statistics testing whether area B predicts area A at each lag and time step.
    bin_centers : ndarray, shape (n_time_steps,)
        Indices of original time bins corresponding to the center of each sliding window.
        Maps columns in a2b_f/b2a_f back to original data time points. 
    """

    n_samples = int(win_size / time_step)  # Convert to integer
    n_trials, n_times = data_A.shape
    n_time_steps = n_times - n_samples + 1  # Number of valid window positions
    
    # Initialize result arrays [n_lags x n_times]
    a2b_f = np.full((len(lags), n_time_steps), np.nan)
    b2a_f = np.full((len(lags), n_time_steps), np.nan)
    
    # Calculate bin centers for each window position
    bin_centers = np.arange(n_time_steps) + n_samples // 2
    
    # Loop over valid window positions
    for t_ix, window_start in enumerate(range(n_time_steps)):
        window_end = window_start + n_samples
       
        # Extract windows from all trials
        a_window = data_A[:, window_start:window_end]  # [trials x window_bins]
        b_window = data_B[:, window_start:window_end]  # [trials x window_bins]
       
        for lag_ix, this_lag in enumerate(lags):
           
            # Test A -> B
            if this_lag > 0:
                a_pred = a_window[:, :-this_lag].flatten()  # Earlier A values
                b_targ = b_window[:, this_lag:].flatten()   # Later B values
            else:
                a_pred = a_window.flatten()
                b_targ = b_window.flatten()
               
            # Test B -> A 
            if this_lag > 0:
                b_pred = b_window[:, :-this_lag].flatten()  # Earlier B values
                a_targ = a_window[:, this_lag:].flatten()   # Later A values
            else:
                b_pred = b_window.flatten()
                a_targ = a_window.flatten()
           
            # Run regressions and compute F-statistics
            if len(a_pred) > 10:  # Minimum data check
                try:
                    X = sm.add_constant(a_pred)
                    model = sm.OLS(b_targ, X).fit()
                    a2b_f[lag_ix, t_ix] = model.fvalue
                except:
                    pass
                   
            if len(b_pred) > 10:
                try:
                    X = sm.add_constant(b_pred)  
                    model = sm.OLS(a_targ, X).fit()
                    b2a_f[lag_ix, t_ix] = model.fvalue
                except:
                    pass
               
    return a2b_f, b2a_f, bin_centers

In [ ]:
# where are the data?
data_dir = 'C:/Users/thome/Documents/PYTHON/OFC-CdN 3 state self control/decoder_output/'

# get their relevant paths
data_files = h5u.find_h5_files(data_dir)

In [ ]:
h5u.list_hdf5_data(data_files[0])

In [ ]:
# initialize lists to accumulate data into
OFC_ch = []
OFC_unch = []
OFC_alt_ch = []
OFC_alt_unch = []
CdN_ch = []
CdN_unch = []
CdN_alt_ch = []
CdN_alt_unch = []
bhv = pd.DataFrame()
OFC_acc = []
CdN_acc = []
subject = []
session = []
OFC_cue_pp = []


# accumulate data from each file

for f_num, this_file in enumerate(data_files):
    
    if 'D' in Path(this_file).stem:
        s = 0
    else:
        s = 1
            
    # Append the data to the lists
    subject.append(np.ones(len(h5u.pull_from_h5(this_file, 'OFC_acc_mean')), )*s)
    session.append(np.ones(len(h5u.pull_from_h5(this_file, 'OFC_acc_mean')), )*f_num)

    bhv = pd.concat([bhv, pd.read_hdf(this_file, key='bhv')], ignore_index=True)
    
    OFC_ch.append(h5u.pull_from_h5(this_file, 'OFC_ch'))
    OFC_unch.append(h5u.pull_from_h5(this_file, 'OFC_unch'))
    OFC_alt_ch.append(h5u.pull_from_h5(this_file, 'OFC_alt_ch'))
    OFC_alt_unch.append(h5u.pull_from_h5(this_file, 'OFC_alt_unch'))
    
    CdN_ch.append(h5u.pull_from_h5(this_file, 'CdN_ch'))
    CdN_unch.append(h5u.pull_from_h5(this_file, 'CdN_unch'))
    CdN_alt_ch.append(h5u.pull_from_h5(this_file, 'CdN_alt_ch'))
    CdN_alt_unch.append(h5u.pull_from_h5(this_file, 'CdN_alt_unch'))
    
    OFC_acc.append(h5u.pull_from_h5(this_file, 'OFC_acc_mean'))
    CdN_acc.append(h5u.pull_from_h5(this_file, 'CdN_acc_mean'))
    
    ts = h5u.pull_from_h5(this_file, 'ts')
    
    
# convert everything back to arrays
subject = np.concatenate(subject, axis=0)
session = np.concatenate(session, axis=0)

OFC_ch = np.concatenate(OFC_ch, axis=0)
OFC_unch = np.concatenate(OFC_unch, axis=0)
OFC_alt_ch = np.concatenate(OFC_alt_ch, axis=0)
OFC_alt_unch = np.concatenate(OFC_alt_unch, axis=0)

CdN_ch = np.concatenate(CdN_ch, axis=0)
CdN_unch = np.concatenate(CdN_unch, axis=0)
CdN_alt_ch = np.concatenate(CdN_alt_ch, axis=0)
CdN_alt_unch = np.concatenate(CdN_alt_unch, axis=0)

OFC_acc = np.concatenate(OFC_acc, axis=0)
CdN_acc = np.concatenate(CdN_acc, axis=0)

In [ ]:
# get the labels associated with each unique state-value pair
ch_val, unch_val = get_ch_and_unch_vals(bhv)
s_ch_val = ch_val.copy()
s_unch_val = unch_val.copy()

s_ch_val[bhv['state'] == 2] = s_ch_val[bhv['state'] == 2] + 4
s_ch_val[bhv['state'] == 3] = s_ch_val[bhv['state'] == 3] + 8
s_unch_val[bhv['state'] == 2] = s_unch_val[bhv['state'] == 2] + 4
s_unch_val[bhv['state'] == 3] = s_unch_val[bhv['state'] == 3] + 8

In [ ]:
# align posteriors to saccades

# set some parameters
n_trials = len(bhv)
ts_offset = [12, 13] # each sample is 25 ms long, so, 12 samples = 300 ms

# initialize lists to accumulate data into

# arrays are organized along 3rd dim as: ch_val, unch_val, alt_ch, alt_unch
first_sacc_ofc_pps = np.full((n_trials, np.sum(ts_offset), 4), np.nan)
first_sacc_cdn_pps = np.full((n_trials, np.sum(ts_offset), 4), np.nan)
second_sacc_ofc_pps = np.full((n_trials, np.sum(ts_offset), 4), np.nan)
second_sacc_cdn_pps = np.full((n_trials, np.sum(ts_offset), 4), np.nan)

# loop over individual trials
for t in tqdm(range(n_trials)):

    # first, only analyze the first saccade

    if bhv['n_sacc'].iloc[t] > 0:

        # get index of the first saccade on this trial
        t_sacc1_time_ix = np.argmin(np.abs(bhv['sacc1_t'].iloc[t] - ts))
        t_sacc1_start =  t_sacc1_time_ix - ts_offset[0]
        ts_sacc1_end = t_sacc1_time_ix + ts_offset[1]

        first_sacc_ofc_pps[t,:,0] = OFC_ch[t, t_sacc1_start:ts_sacc1_end]
        first_sacc_ofc_pps[t,:,1] = OFC_unch[t, t_sacc1_start:ts_sacc1_end]
        first_sacc_ofc_pps[t,:,2] = OFC_alt_ch[t, t_sacc1_start:ts_sacc1_end]
        first_sacc_ofc_pps[t,:,3] = OFC_alt_unch[t, t_sacc1_start:ts_sacc1_end]

        first_sacc_cdn_pps[t,:,0] = CdN_ch[t, t_sacc1_start:ts_sacc1_end]
        first_sacc_cdn_pps[t,:,1] = CdN_unch[t, t_sacc1_start:ts_sacc1_end]
        first_sacc_cdn_pps[t,:,2] = CdN_alt_ch[t, t_sacc1_start:ts_sacc1_end]
        first_sacc_cdn_pps[t,:,3] = CdN_alt_unch[t, t_sacc1_start:ts_sacc1_end]

    # now pull out the second saccade of double-saccade trials
    if bhv['n_sacc'].iloc[t] == 2:

        # get index of the first saccade on this trial
        t_sacc2_time_ix = np.argmin(np.abs(bhv['sacc2_t'].iloc[t] - ts))
        ts_sacc2_start =  t_sacc2_time_ix - ts_offset[0]
        ts_sacc2_end = t_sacc2_time_ix + ts_offset[1]

        if ts_sacc2_end < len(ts):

            second_sacc_ofc_pps[t,:,0] = OFC_ch[t, ts_sacc2_start:ts_sacc2_end]
            second_sacc_ofc_pps[t,:,1] = OFC_unch[t, ts_sacc2_start:ts_sacc2_end]
            second_sacc_ofc_pps[t,:,2] = OFC_alt_ch[t, ts_sacc2_start:ts_sacc2_end]
            second_sacc_ofc_pps[t,:,3] = OFC_alt_unch[t, ts_sacc2_start:ts_sacc2_end]

            second_sacc_cdn_pps[t,:,0] = CdN_ch[t, ts_sacc2_start:ts_sacc2_end]
            second_sacc_cdn_pps[t,:,1] = CdN_unch[t, ts_sacc2_start:ts_sacc2_end]
            second_sacc_cdn_pps[t,:,2] = CdN_alt_ch[t, ts_sacc2_start:ts_sacc2_end]
            second_sacc_cdn_pps[t,:,3] = CdN_alt_unch[t, ts_sacc2_start:ts_sacc2_end]


In [ ]:
# create indices for the single and double saccade trials
single_ix = (bhv['n_sacc'] == 1)
double_ix = (bhv['n_sacc'] == 2)
error_ix = bhv['picked_best'] == 0

# create indices for each subject
d_ix = subject == 0
k_ix = subject == 1

# get means and intervals

#----------------------------
# SUBJECT D
# OFC
# first saccade of single saccade trial
d_1sacc_ofc_ch_mean, d_1sacc_ofc_ch_ci = calculate_mean_and_interval(first_sacc_ofc_pps[single_ix & d_ix,:, 0], 'bootstrap')
d_1sacc_ofc_unch_mean, d_1sacc_ofc_unch_ci = calculate_mean_and_interval(first_sacc_ofc_pps[single_ix & d_ix,:, 1], 'bootstrap')
d_1sacc_ofc_alt_ch_mean, d_1sacc_ofc_alt_ch_ci = calculate_mean_and_interval(first_sacc_ofc_pps[single_ix & d_ix,:, 2], 'bootstrap')
d_1sacc_ofc_alt_unch_mean, d_1sacc_ofc_alt_unch_ci = calculate_mean_and_interval(first_sacc_ofc_pps[single_ix & d_ix,:, 3], 'bootstrap')

# first saccade of double saccade trial
d_1sacc2_ofc_ch_mean, d_1sacc2_ofc_ch_ci = calculate_mean_and_interval(first_sacc_ofc_pps[double_ix & d_ix,:, 0], 'bootstrap')
d_1sacc2_ofc_unch_mean, d_1sacc2_ofc_unch_ci = calculate_mean_and_interval(first_sacc_ofc_pps[double_ix & d_ix,:, 1], 'bootstrap')
d_1sacc2_ofc_alt_ch_mean, d_1sacc2_ofc_alt_ch_ci = calculate_mean_and_interval(first_sacc_ofc_pps[double_ix & d_ix,:, 2], 'bootstrap')
d_1sacc2_ofc_alt_unch_mean, d_1sacc2_ofc_alt_unch_ci = calculate_mean_and_interval(first_sacc_ofc_pps[double_ix & d_ix,:, 3], 'bootstrap')

# second saccade of double saccade trial
d_2sacc2_ofc_ch_mean, d_2sacc2_ofc_ch_ci = calculate_mean_and_interval(second_sacc_ofc_pps[double_ix & d_ix,:, 0], 'bootstrap')
d_2sacc2_ofc_unch_mean, d_2sacc2_ofc_unch_ci = calculate_mean_and_interval(second_sacc_ofc_pps[double_ix & d_ix,:, 1], 'bootstrap')
d_2sacc2_ofc_alt_ch_mean, d_2sacc2_ofc_alt_ch_ci = calculate_mean_and_interval(second_sacc_ofc_pps[double_ix & d_ix,:, 2], 'bootstrap')
d_2sacc2_ofc_alt_unch_mean, d_2sacc2_ofc_alt_unch_ci = calculate_mean_and_interval(second_sacc_ofc_pps[double_ix & d_ix,:, 3], 'bootstrap')

# first saccade of an error
d_error_ofc_ch_mean, d_error_ofc_ch_ci = calculate_mean_and_interval(first_sacc_ofc_pps[single_ix & d_ix & error_ix,:, 0], 'bootstrap')
d_error_ofc_unch_mean, d_error_ofc_unch_ci = calculate_mean_and_interval(first_sacc_ofc_pps[single_ix & d_ix & error_ix,:, 1], 'bootstrap')
d_error_ofc_alt_ch_mean, d_error_ofc_alt_ch_ci = calculate_mean_and_interval(first_sacc_ofc_pps[single_ix & d_ix & error_ix,:, 2], 'bootstrap')
d_error_ofc_alt_unch_mean, d_error_ofc_alt_unch_ci = calculate_mean_and_interval(first_sacc_ofc_pps[single_ix & d_ix & error_ix,:, 3], 'bootstrap')

# CDN
# first saccade of single saccade trial
d_1sacc_cdn_ch_mean, d_1sacc_cdn_ch_ci = calculate_mean_and_interval(first_sacc_cdn_pps[single_ix & d_ix,:, 0], 'bootstrap')
d_1sacc_cdn_unch_mean, d_1sacc_cdn_unch_ci = calculate_mean_and_interval(first_sacc_cdn_pps[single_ix & d_ix,:, 1], 'bootstrap')
d_1sacc_cdn_alt_ch_mean, d_1sacc_cdn_alt_ch_ci = calculate_mean_and_interval(first_sacc_cdn_pps[single_ix & d_ix,:, 2], 'bootstrap')
d_1sacc_cdn_alt_unch_mean, d_1sacc_cdn_alt_unch_ci = calculate_mean_and_interval(first_sacc_cdn_pps[single_ix & d_ix,:, 3], 'bootstrap')

# first saccade of double saccade trial
d_1sacc2_cdn_ch_mean, d_1sacc2_cdn_ch_ci = calculate_mean_and_interval(first_sacc_cdn_pps[double_ix & d_ix,:, 0], 'bootstrap')
d_1sacc2_cdn_unch_mean, d_1sacc2_cdn_unch_ci = calculate_mean_and_interval(first_sacc_cdn_pps[double_ix & d_ix,:, 1], 'bootstrap')
d_1sacc2_cdn_alt_ch_mean, d_1sacc2_cdn_alt_ch_ci = calculate_mean_and_interval(first_sacc_cdn_pps[double_ix & d_ix,:, 2], 'bootstrap')
d_1sacc2_cdn_alt_unch_mean, d_1sacc2_cdn_alt_unch_ci = calculate_mean_and_interval(first_sacc_cdn_pps[double_ix & d_ix,:, 3], 'bootstrap')

# second saccade of double saccade trial
d_2sacc2_cdn_ch_mean, d_2sacc2_cdn_ch_ci = calculate_mean_and_interval(second_sacc_cdn_pps[double_ix & d_ix,:, 0], 'bootstrap')
d_2sacc2_cdn_unch_mean, d_2sacc2_cdn_unch_ci = calculate_mean_and_interval(second_sacc_cdn_pps[double_ix & d_ix,:, 1], 'bootstrap')
d_2sacc2_cdn_alt_ch_mean, d_2sacc2_cdn_alt_ch_ci = calculate_mean_and_interval(second_sacc_cdn_pps[double_ix & d_ix,:, 2], 'bootstrap')
d_2sacc2_cdn_alt_unch_mean, d_2sacc2_cdn_alt_unch_ci = calculate_mean_and_interval(second_sacc_cdn_pps[double_ix & d_ix,:, 3], 'bootstrap')

# first saccade of an error
d_error_cdn_ch_mean, d_error_cdn_ch_ci = calculate_mean_and_interval(first_sacc_cdn_pps[single_ix & d_ix & error_ix,:, 0], 'bootstrap')
d_error_cdn_unch_mean, d_error_cdn_unch_ci = calculate_mean_and_interval(first_sacc_cdn_pps[single_ix & d_ix & error_ix,:, 1], 'bootstrap')
d_error_cdn_alt_ch_mean, d_error_cdn_alt_ch_ci = calculate_mean_and_interval(first_sacc_cdn_pps[single_ix & d_ix & error_ix,:, 2], 'bootstrap')
d_error_cdn_alt_unch_mean, d_error_cdn_alt_unch_ci = calculate_mean_and_interval(first_sacc_cdn_pps[single_ix & d_ix & error_ix,:, 3], 'bootstrap')

#----------------------------
# SUBJECT K
# OFC
# first saccade of single saccade trial
k_1sacc_ofc_ch_mean, k_1sacc_ofc_ch_ci = calculate_mean_and_interval(first_sacc_ofc_pps[single_ix & k_ix,:, 0], 'bootstrap')
k_1sacc_ofc_unch_mean, k_1sacc_ofc_unch_ci = calculate_mean_and_interval(first_sacc_ofc_pps[single_ix & k_ix,:, 1], 'bootstrap')
k_1sacc_ofc_alt_ch_mean, k_1sacc_ofc_alt_ch_ci = calculate_mean_and_interval(first_sacc_ofc_pps[single_ix & k_ix,:, 2], 'bootstrap')
k_1sacc_ofc_alt_unch_mean, k_1sacc_ofc_alt_unch_ci = calculate_mean_and_interval(first_sacc_ofc_pps[single_ix & k_ix,:, 3], 'bootstrap')

# first saccade of double saccade trial
k_1sacc2_ofc_ch_mean, k_1sacc2_ofc_ch_ci = calculate_mean_and_interval(first_sacc_ofc_pps[double_ix & k_ix,:, 0], 'bootstrap')
k_1sacc2_ofc_unch_mean, k_1sacc2_ofc_unch_ci = calculate_mean_and_interval(first_sacc_ofc_pps[double_ix & k_ix,:, 1], 'bootstrap')
k_1sacc2_ofc_alt_ch_mean, k_1sacc2_ofc_alt_ch_ci = calculate_mean_and_interval(first_sacc_ofc_pps[double_ix & k_ix,:, 2], 'bootstrap')
k_1sacc2_ofc_alt_unch_mean, k_1sacc2_ofc_alt_unch_ci = calculate_mean_and_interval(first_sacc_ofc_pps[double_ix & k_ix,:, 3], 'bootstrap')

# second saccade of double saccade trial
k_2sacc2_ofc_ch_mean, k_2sacc2_ofc_ch_ci = calculate_mean_and_interval(second_sacc_ofc_pps[double_ix & k_ix,:, 0], 'bootstrap')
k_2sacc2_ofc_unch_mean, k_2sacc2_ofc_unch_ci = calculate_mean_and_interval(second_sacc_ofc_pps[double_ix & k_ix,:, 1], 'bootstrap')
k_2sacc2_ofc_alt_ch_mean, k_2sacc2_ofc_alt_ch_ci = calculate_mean_and_interval(second_sacc_ofc_pps[double_ix & k_ix,:, 2], 'bootstrap')
k_2sacc2_ofc_alt_unch_mean, k_2sacc2_ofc_alt_unch_ci = calculate_mean_and_interval(second_sacc_ofc_pps[double_ix & k_ix,:, 3], 'bootstrap')

# first saccade of an error
k_error_ofc_ch_mean, k_error_ofc_ch_ci = calculate_mean_and_interval(first_sacc_ofc_pps[single_ix & k_ix & error_ix,:, 0], 'bootstrap')
k_error_ofc_unch_mean, k_error_ofc_unch_ci = calculate_mean_and_interval(first_sacc_ofc_pps[single_ix & k_ix & error_ix,:, 1], 'bootstrap')
k_error_ofc_alt_ch_mean, k_error_ofc_alt_ch_ci = calculate_mean_and_interval(first_sacc_ofc_pps[single_ix & k_ix & error_ix,:, 2], 'bootstrap')
k_error_ofc_alt_unch_mean, k_error_ofc_alt_unch_ci = calculate_mean_and_interval(first_sacc_ofc_pps[single_ix & k_ix & error_ix,:, 3], 'bootstrap')

# CDN
# first saccade of single saccade trial
k_1sacc_cdn_ch_mean, k_1sacc_cdn_ch_ci = calculate_mean_and_interval(first_sacc_cdn_pps[single_ix & k_ix,:, 0], 'bootstrap')
k_1sacc_cdn_unch_mean, k_1sacc_cdn_unch_ci = calculate_mean_and_interval(first_sacc_cdn_pps[single_ix & k_ix,:, 1], 'bootstrap')
k_1sacc_cdn_alt_ch_mean, k_1sacc_cdn_alt_ch_ci = calculate_mean_and_interval(first_sacc_cdn_pps[single_ix & k_ix,:, 2], 'bootstrap')
k_1sacc_cdn_alt_unch_mean, k_1sacc_cdn_alt_unch_ci = calculate_mean_and_interval(first_sacc_cdn_pps[single_ix & k_ix,:, 3], 'bootstrap')

# first saccade of double saccade trial
k_1sacc2_cdn_ch_mean, k_1sacc2_cdn_ch_ci = calculate_mean_and_interval(first_sacc_cdn_pps[double_ix & k_ix,:, 0], 'bootstrap')
k_1sacc2_cdn_unch_mean, k_1sacc2_cdn_unch_ci = calculate_mean_and_interval(first_sacc_cdn_pps[double_ix & k_ix,:, 1], 'bootstrap')
k_1sacc2_cdn_alt_ch_mean, k_1sacc2_cdn_alt_ch_ci = calculate_mean_and_interval(first_sacc_cdn_pps[double_ix & k_ix,:, 2], 'bootstrap')
k_1sacc2_cdn_alt_unch_mean, k_1sacc2_cdn_alt_unch_ci = calculate_mean_and_interval(first_sacc_cdn_pps[double_ix & k_ix,:, 3], 'bootstrap')

# second saccade of double saccade trial
k_2sacc2_cdn_ch_mean, k_2sacc2_cdn_ch_ci = calculate_mean_and_interval(second_sacc_cdn_pps[double_ix & k_ix,:, 0], 'bootstrap')
k_2sacc2_cdn_unch_mean, k_2sacc2_cdn_unch_ci = calculate_mean_and_interval(second_sacc_cdn_pps[double_ix & k_ix,:, 1], 'bootstrap')
k_2sacc2_cdn_alt_ch_mean, k_2sacc2_cdn_alt_ch_ci = calculate_mean_and_interval(second_sacc_cdn_pps[double_ix & k_ix,:, 2], 'bootstrap')
k_2sacc2_cdn_alt_unch_mean, k_2sacc2_cdn_alt_unch_ci = calculate_mean_and_interval(second_sacc_cdn_pps[double_ix & k_ix,:, 3], 'bootstrap')

# first saccade of an error
k_error_cdn_ch_mean, k_error_cdn_ch_ci = calculate_mean_and_interval(first_sacc_cdn_pps[single_ix & k_ix & error_ix,:, 0], 'bootstrap')
k_error_cdn_unch_mean, k_error_cdn_unch_ci = calculate_mean_and_interval(first_sacc_cdn_pps[single_ix & k_ix & error_ix,:, 1], 'bootstrap')
k_error_cdn_alt_ch_mean, k_error_cdn_alt_ch_ci = calculate_mean_and_interval(first_sacc_cdn_pps[single_ix & k_ix & error_ix,:, 2], 'bootstrap')
k_error_cdn_alt_unch_mean, k_error_cdn_alt_unch_ci = calculate_mean_and_interval(first_sacc_cdn_pps[single_ix & k_ix & error_ix,:, 3], 'bootstrap')

In [ ]:
# do some plotting

# create array for the x-axis
ts_step = np.mean(np.diff(ts))
sacc_ts = np.arange(0, np.sum(ts_offset)*ts_step, ts_step) - ts_offset[0]*ts_step

fig, ax = plt.subplots(4, 3, figsize=(8, 10))
plt.subplots_adjust(wspace=0.4, hspace=0.4)  # add some white space

ch_col = 'tab:red'
unch_col = 'tab:blue'
alt_ch_col = 'tab:gray'
alt_unch_col = 'black'
alpha = .3

# SUBJECT D - OFC

# first saccade of single saccade trial
ax[0,0].fill_between(sacc_ts, d_1sacc_ofc_ch_mean - d_1sacc_ofc_ch_ci, d_1sacc_ofc_ch_mean + d_1sacc_ofc_ch_ci, color=ch_col, alpha=alpha)
ax[0,0].plot(sacc_ts, d_1sacc_ofc_ch_mean, color = ch_col)
ax[0,0].fill_between(sacc_ts, d_1sacc_ofc_unch_mean - d_1sacc_ofc_unch_ci, d_1sacc_ofc_unch_mean + d_1sacc_ofc_unch_ci, color=unch_col, alpha=alpha)
ax[0,0].plot(sacc_ts, d_1sacc_ofc_unch_mean, color = unch_col)
ax[0,0].fill_between(sacc_ts, d_1sacc_ofc_alt_ch_mean - d_1sacc_ofc_alt_ch_ci, d_1sacc_ofc_alt_ch_mean + d_1sacc_ofc_alt_ch_ci, color=alt_ch_col, alpha=alpha)
ax[0,0].plot(sacc_ts, d_1sacc_ofc_alt_ch_mean, color = alt_ch_col)
ax[0,0].fill_between(sacc_ts, d_1sacc_ofc_alt_unch_mean - d_1sacc_ofc_alt_unch_ci, d_1sacc_ofc_alt_unch_mean + d_1sacc_ofc_alt_unch_ci, color=alt_unch_col, alpha=alpha)
ax[0,0].plot(sacc_ts, d_1sacc_ofc_alt_unch_mean, color = alt_unch_col)
ax[0,0].set_ylim((0, .8))

# first saccade of double saccade trial
ax[0,1].fill_between(sacc_ts, d_1sacc2_ofc_ch_mean - d_1sacc2_ofc_ch_ci, d_1sacc2_ofc_ch_mean + d_1sacc2_ofc_ch_ci, color=ch_col, alpha=alpha)
ax[0,1].plot(sacc_ts, d_1sacc2_ofc_ch_mean, color = ch_col)

ax[0,1].fill_between(sacc_ts, d_1sacc2_ofc_unch_mean - d_1sacc2_ofc_unch_ci, d_1sacc2_ofc_unch_mean + d_1sacc2_ofc_unch_ci, color=unch_col, alpha=alpha)
ax[0,1].plot(sacc_ts, d_1sacc2_ofc_unch_mean, color = unch_col)

ax[0,1].fill_between(sacc_ts, d_1sacc2_ofc_alt_ch_mean - d_1sacc2_ofc_alt_ch_ci, d_1sacc2_ofc_alt_ch_mean + d_1sacc2_ofc_alt_ch_ci, color=alt_ch_col, alpha=alpha)
ax[0,1].plot(sacc_ts, d_1sacc2_ofc_alt_ch_mean, color = alt_ch_col)

ax[0,1].fill_between(sacc_ts, d_1sacc2_ofc_alt_unch_mean - d_1sacc2_ofc_alt_unch_ci, d_1sacc2_ofc_alt_unch_mean + d_1sacc2_ofc_alt_unch_ci, color=alt_unch_col, alpha=alpha)
ax[0,1].plot(sacc_ts, d_1sacc2_ofc_alt_unch_mean, color = alt_unch_col)
ax[0,1].set_ylim((0, .8))

# second saccade of double saccade trial
ax[0,2].fill_between(sacc_ts, d_2sacc2_ofc_ch_mean - d_2sacc2_ofc_ch_ci, d_2sacc2_ofc_ch_mean + d_2sacc2_ofc_ch_ci, color=ch_col, alpha=alpha)
ax[0,2].plot(sacc_ts, d_2sacc2_ofc_ch_mean, color = ch_col)

ax[0,2].fill_between(sacc_ts, d_2sacc2_ofc_unch_mean - d_2sacc2_ofc_unch_ci, d_2sacc2_ofc_unch_mean + d_2sacc2_ofc_unch_ci, color=unch_col, alpha=alpha)
ax[0,2].plot(sacc_ts, d_2sacc2_ofc_unch_mean, color = unch_col)

ax[0,2].fill_between(sacc_ts, d_2sacc2_ofc_alt_ch_mean - d_2sacc2_ofc_alt_ch_ci, d_2sacc2_ofc_alt_ch_mean + d_2sacc2_ofc_alt_ch_ci, color=alt_ch_col, alpha=alpha)
ax[0,2].plot(sacc_ts, d_2sacc2_ofc_alt_ch_mean, color = alt_ch_col)

ax[0,2].fill_between(sacc_ts, d_2sacc2_ofc_alt_unch_mean - d_2sacc2_ofc_alt_unch_ci, d_2sacc2_ofc_alt_unch_mean + d_2sacc2_ofc_alt_unch_ci, color=alt_unch_col, alpha=alpha)
ax[0,2].plot(sacc_ts, d_2sacc2_ofc_alt_unch_mean, color = alt_unch_col)
ax[0,2].set_ylim((0, .8))

# SUBJECT D - CDN

# first saccade of single saccade trial
ax[1,0].fill_between(sacc_ts, d_1sacc_cdn_ch_mean - d_1sacc_cdn_ch_ci, d_1sacc_cdn_ch_mean + d_1sacc_cdn_ch_ci, color=ch_col, alpha=alpha)
ax[1,0].plot(sacc_ts, d_1sacc_cdn_ch_mean, color = ch_col)

ax[1,0].fill_between(sacc_ts, d_1sacc_cdn_unch_mean - d_1sacc_cdn_unch_ci, d_1sacc_cdn_unch_mean + d_1sacc_cdn_unch_ci, color=unch_col, alpha=alpha)
ax[1,0].plot(sacc_ts, d_1sacc_cdn_unch_mean, color = unch_col)

ax[1,0].fill_between(sacc_ts, d_1sacc_cdn_alt_ch_mean - d_1sacc_cdn_alt_ch_ci, d_1sacc_cdn_alt_ch_mean + d_1sacc_cdn_alt_ch_ci, color=alt_ch_col, alpha=alpha)
ax[1,0].plot(sacc_ts, d_1sacc_cdn_alt_ch_mean, color = alt_ch_col)

ax[1,0].fill_between(sacc_ts, d_1sacc_cdn_alt_unch_mean - d_1sacc_cdn_alt_unch_ci, d_1sacc_cdn_alt_unch_mean + d_1sacc_cdn_alt_unch_ci, color=alt_unch_col, alpha=alpha)
ax[1,0].plot(sacc_ts, d_1sacc_cdn_alt_unch_mean, color = alt_unch_col)
ax[1,0].set_ylim((0, .6))

# first saccade of double saccade trial
ax[1,1].fill_between(sacc_ts, d_1sacc2_cdn_ch_mean - d_1sacc2_cdn_ch_ci, d_1sacc2_cdn_ch_mean + d_1sacc2_cdn_ch_ci, color=ch_col, alpha=alpha)
ax[1,1].plot(sacc_ts, d_1sacc2_cdn_ch_mean, color = ch_col)

ax[1,1].fill_between(sacc_ts, d_1sacc2_cdn_unch_mean - d_1sacc2_cdn_unch_ci, d_1sacc2_cdn_unch_mean + d_1sacc2_cdn_unch_ci, color=unch_col, alpha=alpha)
ax[1,1].plot(sacc_ts, d_1sacc2_cdn_unch_mean, color = unch_col)

ax[1,1].fill_between(sacc_ts, d_1sacc2_cdn_alt_ch_mean - d_1sacc2_cdn_alt_ch_ci, d_1sacc2_cdn_alt_ch_mean + d_1sacc2_cdn_alt_ch_ci, color=alt_ch_col, alpha=alpha)
ax[1,1].plot(sacc_ts, d_1sacc2_cdn_alt_ch_mean, color = alt_ch_col)

ax[1,1].fill_between(sacc_ts, d_1sacc2_cdn_alt_unch_mean - d_1sacc2_cdn_alt_unch_ci, d_1sacc2_cdn_alt_unch_mean + d_1sacc2_cdn_alt_unch_ci, color=alt_unch_col, alpha=alpha)
ax[1,1].plot(sacc_ts, d_1sacc2_cdn_alt_unch_mean, color = alt_unch_col)
ax[1,1].set_ylim((0, .6))

# second saccade of double saccade trial
ax[1,2].fill_between(sacc_ts, d_2sacc2_cdn_ch_mean - d_2sacc2_cdn_ch_ci, d_2sacc2_cdn_ch_mean + d_2sacc2_cdn_ch_ci, color=ch_col, alpha=alpha)
ax[1,2].plot(sacc_ts, d_2sacc2_cdn_ch_mean, color = ch_col)

ax[1,2].fill_between(sacc_ts, d_2sacc2_cdn_unch_mean - d_2sacc2_cdn_unch_ci, d_2sacc2_cdn_unch_mean + d_2sacc2_cdn_unch_ci, color=unch_col, alpha=alpha)
ax[1,2].plot(sacc_ts, d_2sacc2_cdn_unch_mean, color = unch_col)

ax[1,2].fill_between(sacc_ts, d_2sacc2_cdn_alt_ch_mean - d_2sacc2_cdn_alt_ch_ci, d_2sacc2_cdn_alt_ch_mean + d_2sacc2_cdn_alt_ch_ci, color=alt_ch_col, alpha=alpha)
ax[1,2].plot(sacc_ts, d_2sacc2_cdn_alt_ch_mean, color = alt_ch_col)

ax[1,2].fill_between(sacc_ts, d_2sacc2_cdn_alt_unch_mean - d_2sacc2_cdn_alt_unch_ci, d_2sacc2_cdn_alt_unch_mean + d_2sacc2_cdn_alt_unch_ci, color=alt_unch_col, alpha=alpha)
ax[1,2].plot(sacc_ts, d_2sacc2_cdn_alt_unch_mean, color = alt_unch_col)
ax[1,2].set_ylim((0, .6))

# SUBJECT K - OFC

# first saccade of single saccade trial
ax[2,0].fill_between(sacc_ts, k_1sacc_ofc_ch_mean - k_1sacc_ofc_ch_ci, k_1sacc_ofc_ch_mean + k_1sacc_ofc_ch_ci, color=ch_col, alpha=alpha)
ax[2,0].plot(sacc_ts, k_1sacc_ofc_ch_mean, color = ch_col)

ax[2,0].fill_between(sacc_ts, k_1sacc_ofc_unch_mean - k_1sacc_ofc_unch_ci, k_1sacc_ofc_unch_mean + k_1sacc_ofc_unch_ci, color=unch_col, alpha=alpha)
ax[2,0].plot(sacc_ts, k_1sacc_ofc_unch_mean, color = unch_col)

ax[2,0].fill_between(sacc_ts, k_1sacc_ofc_alt_ch_mean - k_1sacc_ofc_alt_ch_ci, k_1sacc_ofc_alt_ch_mean + k_1sacc_ofc_alt_ch_ci, color=alt_ch_col, alpha=alpha)
ax[2,0].plot(sacc_ts, k_1sacc_ofc_alt_ch_mean, color = alt_ch_col)

ax[2,0].fill_between(sacc_ts, k_1sacc_ofc_alt_unch_mean - k_1sacc_ofc_alt_unch_ci, k_1sacc_ofc_alt_unch_mean + k_1sacc_ofc_alt_unch_ci, color=alt_unch_col, alpha=alpha)
ax[2,0].plot(sacc_ts, k_1sacc_ofc_alt_unch_mean, color = alt_unch_col)
ax[2,0].set_ylim((0, .9))
ax[2,0].set_yticks((0, .3, .6, .9))

# first saccade of double saccade trial
ax[2,1].fill_between(sacc_ts, k_1sacc2_ofc_ch_mean - k_1sacc2_ofc_ch_ci, k_1sacc2_ofc_ch_mean + k_1sacc2_ofc_ch_ci, color=ch_col, alpha=alpha)
ax[2,1].plot(sacc_ts, k_1sacc2_ofc_ch_mean, color = ch_col)

ax[2,1].fill_between(sacc_ts, k_1sacc2_ofc_unch_mean - k_1sacc2_ofc_unch_ci, k_1sacc2_ofc_unch_mean + k_1sacc2_ofc_unch_ci, color=unch_col, alpha=alpha)
ax[2,1].plot(sacc_ts, k_1sacc2_ofc_unch_mean, color = unch_col)

ax[2,1].fill_between(sacc_ts, k_1sacc2_ofc_alt_ch_mean - k_1sacc2_ofc_alt_ch_ci, k_1sacc2_ofc_alt_ch_mean + k_1sacc2_ofc_alt_ch_ci, color=alt_ch_col, alpha=alpha)
ax[2,1].plot(sacc_ts, k_1sacc2_ofc_alt_ch_mean, color = alt_ch_col)

ax[2,1].fill_between(sacc_ts, k_1sacc2_ofc_alt_unch_mean - k_1sacc2_ofc_alt_unch_ci, k_1sacc2_ofc_alt_unch_mean + k_1sacc2_ofc_alt_unch_ci, color=alt_unch_col, alpha=alpha)
ax[2,1].plot(sacc_ts, k_1sacc2_ofc_alt_unch_mean, color = alt_unch_col)
ax[2,1].set_ylim((0, .9))
ax[2,1].set_yticks((0, .3, .6, .9))

# second saccade of double saccade trial
ax[2,2].fill_between(sacc_ts, k_2sacc2_ofc_ch_mean - k_2sacc2_ofc_ch_ci, k_2sacc2_ofc_ch_mean + k_2sacc2_ofc_ch_ci, color=ch_col, alpha=alpha)
ax[2,2].plot(sacc_ts, k_2sacc2_ofc_ch_mean, color = ch_col)

ax[2,2].fill_between(sacc_ts, k_2sacc2_ofc_unch_mean - k_2sacc2_ofc_unch_ci, k_2sacc2_ofc_unch_mean + k_2sacc2_ofc_unch_ci, color=unch_col, alpha=alpha)
ax[2,2].plot(sacc_ts, k_2sacc2_ofc_unch_mean, color = unch_col)

ax[2,2].fill_between(sacc_ts, k_2sacc2_ofc_alt_ch_mean - k_2sacc2_ofc_alt_ch_ci, k_2sacc2_ofc_alt_ch_mean + k_2sacc2_ofc_alt_ch_ci, color=alt_ch_col, alpha=alpha)
ax[2,2].plot(sacc_ts, k_2sacc2_ofc_alt_ch_mean, color = alt_ch_col)

ax[2,2].fill_between(sacc_ts, k_2sacc2_ofc_alt_unch_mean - k_2sacc2_ofc_alt_unch_ci, k_2sacc2_ofc_alt_unch_mean + k_2sacc2_ofc_alt_unch_ci, color=alt_unch_col, alpha=alpha)
ax[2,2].plot(sacc_ts, k_2sacc2_ofc_alt_unch_mean, color = alt_unch_col)
ax[2,2].set_ylim((0, .9))
ax[2,2].set_yticks((0, .3, .6, .9))

# SUBJECT D - CDN

# first saccade of single saccade trial
ax[3,0].fill_between(sacc_ts, k_1sacc_cdn_ch_mean - k_1sacc_cdn_ch_ci, k_1sacc_cdn_ch_mean + k_1sacc_cdn_ch_ci, color=ch_col, alpha=alpha)
ax[3,0].plot(sacc_ts, k_1sacc_cdn_ch_mean, color = ch_col)

ax[3,0].fill_between(sacc_ts, k_1sacc_cdn_unch_mean - k_1sacc_cdn_unch_ci, k_1sacc_cdn_unch_mean + k_1sacc_cdn_unch_ci, color=unch_col, alpha=alpha)
ax[3,0].plot(sacc_ts, k_1sacc_cdn_unch_mean, color = unch_col)

ax[3,0].fill_between(sacc_ts, k_1sacc_cdn_alt_ch_mean - k_1sacc_cdn_alt_ch_ci, k_1sacc_cdn_alt_ch_mean + k_1sacc_cdn_alt_ch_ci, color=alt_ch_col, alpha=alpha)
ax[3,0].plot(sacc_ts, k_1sacc_cdn_alt_ch_mean, color = alt_ch_col)

ax[3,0].fill_between(sacc_ts, k_1sacc_cdn_alt_unch_mean - k_1sacc_cdn_alt_unch_ci, k_1sacc_cdn_alt_unch_mean + k_1sacc_cdn_alt_unch_ci, color=alt_unch_col, alpha=alpha)
ax[3,0].plot(sacc_ts, k_1sacc_cdn_alt_unch_mean, color = alt_unch_col)
ax[3,0].set_ylim((0, .5))

# first saccade of double saccade trial
ax[3,1].fill_between(sacc_ts, k_1sacc2_cdn_ch_mean - k_1sacc2_cdn_ch_ci, k_1sacc2_cdn_ch_mean + k_1sacc2_cdn_ch_ci, color=ch_col, alpha=alpha)
ax[3,1].plot(sacc_ts, k_1sacc2_cdn_ch_mean, color = ch_col)

ax[3,1].fill_between(sacc_ts, k_1sacc2_cdn_unch_mean - k_1sacc2_cdn_unch_ci, k_1sacc2_cdn_unch_mean + k_1sacc2_cdn_unch_ci, color=unch_col, alpha=alpha)
ax[3,1].plot(sacc_ts, k_1sacc2_cdn_unch_mean, color = unch_col)

ax[3,1].fill_between(sacc_ts, k_1sacc2_cdn_alt_ch_mean - k_1sacc2_cdn_alt_ch_ci, k_1sacc2_cdn_alt_ch_mean + k_1sacc2_cdn_alt_ch_ci, color=alt_ch_col, alpha=alpha)
ax[3,1].plot(sacc_ts, k_1sacc2_cdn_alt_ch_mean, color = alt_ch_col)

ax[3,1].fill_between(sacc_ts, k_1sacc2_cdn_alt_unch_mean - k_1sacc2_cdn_alt_unch_ci, k_1sacc2_cdn_alt_unch_mean + k_1sacc2_cdn_alt_unch_ci, color=alt_unch_col, alpha=alpha)
ax[3,1].plot(sacc_ts, k_1sacc2_cdn_alt_unch_mean, color = alt_unch_col)
ax[3,1].set_ylim((0, .5))

# second saccade of double saccade trial
ax[3,2].fill_between(sacc_ts, k_2sacc2_cdn_ch_mean - k_2sacc2_cdn_ch_ci, k_2sacc2_cdn_ch_mean + k_2sacc2_cdn_ch_ci, color=ch_col, alpha=alpha)
ax[3,2].plot(sacc_ts, k_2sacc2_cdn_ch_mean, color = ch_col)

ax[3,2].fill_between(sacc_ts, k_2sacc2_cdn_unch_mean - k_2sacc2_cdn_unch_ci, k_2sacc2_cdn_unch_mean + k_2sacc2_cdn_unch_ci, color=unch_col, alpha=alpha)
ax[3,2].plot(sacc_ts, k_2sacc2_cdn_unch_mean, color = unch_col)

ax[3,2].fill_between(sacc_ts, k_2sacc2_cdn_alt_ch_mean - k_2sacc2_cdn_alt_ch_ci, k_2sacc2_cdn_alt_ch_mean + k_2sacc2_cdn_alt_ch_ci, color=alt_ch_col, alpha=alpha)
ax[3,2].plot(sacc_ts, k_2sacc2_cdn_alt_ch_mean, color = alt_ch_col)

ax[3,2].fill_between(sacc_ts, k_2sacc2_cdn_alt_unch_mean - k_2sacc2_cdn_alt_unch_ci, k_2sacc2_cdn_alt_unch_mean + k_2sacc2_cdn_alt_unch_ci, color=alt_unch_col, alpha=alpha)
ax[3,2].plot(sacc_ts, k_2sacc2_cdn_alt_unch_mean, color = alt_unch_col)
ax[3,2].set_ylim((0, .5))

# save the figure
#fig.savefig('saccade_aligned_posteriors.svg')

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(8, 5))
plt.subplots_adjust(wspace=0.4, hspace=0.4)  # add some white space

ofc_ch_color = 'tab:red'
cdn_ch_color = 'tab:orange'

ofc_alt_unch_color = 'tab:blue'
cdn_alt_unch_color = 'tab:purple'

subject_d_y_ylims = (0, .7)
subject_k_y_ylims = (0, .9)


# SUBJECT D

# first saccade of single saccade trial
ax[0,0].fill_between(sacc_ts, d_1sacc_ofc_ch_mean - d_1sacc_ofc_ch_ci, d_1sacc_ofc_ch_mean + d_1sacc_ofc_ch_ci, color=ofc_ch_color, alpha=alpha)
ax[0,0].plot(sacc_ts, d_1sacc_ofc_ch_mean, color = ofc_ch_color)

ax[0,0].fill_between(sacc_ts, d_1sacc_cdn_ch_mean - d_1sacc_cdn_ch_ci, d_1sacc_cdn_ch_mean + d_1sacc_cdn_ch_ci, color=cdn_ch_color, alpha=alpha)
ax[0,0].plot(sacc_ts, d_1sacc_cdn_ch_mean, color = cdn_ch_color)


ax[0,0].fill_between(sacc_ts, d_1sacc_ofc_alt_unch_mean - d_1sacc_ofc_alt_unch_ci, d_1sacc_ofc_alt_unch_mean + d_1sacc_ofc_alt_unch_ci, color=ofc_alt_unch_color, alpha=alpha)
ax[0,0].plot(sacc_ts, d_1sacc_ofc_alt_unch_mean, color = ofc_alt_unch_color)

ax[0,0].fill_between(sacc_ts, d_1sacc_cdn_alt_unch_mean - d_1sacc_cdn_alt_unch_ci, d_1sacc_cdn_alt_unch_mean + d_1sacc_cdn_alt_unch_ci, color=cdn_alt_unch_color, alpha=alpha)
ax[0,0].plot(sacc_ts, d_1sacc_cdn_alt_unch_mean, color = cdn_alt_unch_color)

ax[0,0].set_ylim(subject_d_y_ylims)


# first saccade of double saccade trial
ax[0,1].fill_between(sacc_ts, d_1sacc2_ofc_ch_mean - d_1sacc2_ofc_ch_ci, d_1sacc2_ofc_ch_mean + d_1sacc2_ofc_ch_ci, color=ofc_ch_color, alpha=alpha)
ax[0,1].plot(sacc_ts, d_1sacc2_ofc_ch_mean, color = ofc_ch_color)

ax[0,1].fill_between(sacc_ts, d_1sacc2_cdn_ch_mean - d_1sacc2_cdn_ch_ci, d_1sacc2_cdn_ch_mean + d_1sacc2_cdn_ch_ci, color=cdn_ch_color, alpha=alpha)
ax[0,1].plot(sacc_ts, d_1sacc2_cdn_ch_mean, color = cdn_ch_color)

ax[0,1].fill_between(sacc_ts, d_1sacc2_ofc_alt_unch_mean - d_1sacc2_ofc_alt_unch_ci, d_1sacc2_ofc_alt_unch_mean + d_1sacc2_ofc_alt_unch_ci, color=ofc_alt_unch_color, alpha=alpha)
ax[0,1].plot(sacc_ts, d_1sacc2_ofc_alt_unch_mean, color = ofc_alt_unch_color)

ax[0,1].fill_between(sacc_ts, d_1sacc2_cdn_alt_unch_mean - d_1sacc2_cdn_alt_unch_ci, d_1sacc2_cdn_alt_unch_mean + d_1sacc2_cdn_alt_unch_ci, color=cdn_alt_unch_color, alpha=alpha)
ax[0,1].plot(sacc_ts, d_1sacc2_cdn_alt_unch_mean, color = cdn_alt_unch_color)

ax[0,1].set_ylim(subject_d_y_ylims)



# first saccade of double saccade trial
ax[0,2].fill_between(sacc_ts, d_2sacc2_ofc_ch_mean - d_2sacc2_ofc_ch_ci, d_2sacc2_ofc_ch_mean + d_2sacc2_ofc_ch_ci, color=ofc_ch_color, alpha=alpha)
ax[0,2].plot(sacc_ts, d_2sacc2_ofc_ch_mean, color = ofc_ch_color)

ax[0,2].fill_between(sacc_ts, d_2sacc2_cdn_ch_mean - d_2sacc2_cdn_ch_ci, d_2sacc2_cdn_ch_mean + d_2sacc2_cdn_ch_ci, color=cdn_ch_color, alpha=alpha)
ax[0,2].plot(sacc_ts, d_2sacc2_cdn_ch_mean, color = cdn_ch_color)

ax[0,2].fill_between(sacc_ts, d_2sacc2_ofc_alt_unch_mean - d_2sacc2_ofc_alt_unch_ci, d_2sacc2_ofc_alt_unch_mean + d_2sacc2_ofc_alt_unch_ci, color=ofc_alt_unch_color, alpha=alpha)
ax[0,2].plot(sacc_ts, d_2sacc2_ofc_alt_unch_mean, color = ofc_alt_unch_color)

ax[0,2].fill_between(sacc_ts, d_2sacc2_cdn_alt_unch_mean - d_2sacc2_cdn_alt_unch_ci, d_2sacc2_cdn_alt_unch_mean + d_2sacc2_cdn_alt_unch_ci, color=cdn_alt_unch_color, alpha=alpha)
ax[0,2].plot(sacc_ts, d_2sacc2_cdn_alt_unch_mean, color = cdn_alt_unch_color)

ax[0,2].set_ylim(subject_d_y_ylims)


# SUBJECT K
# first saccade of single saccade trial
ax[1,0].fill_between(sacc_ts, k_1sacc_ofc_ch_mean - k_1sacc_ofc_ch_ci, k_1sacc_ofc_ch_mean + k_1sacc_ofc_ch_ci, color=ofc_ch_color, alpha=alpha)
ax[1,0].plot(sacc_ts, k_1sacc_ofc_ch_mean, color = ofc_ch_color)

ax[1,0].fill_between(sacc_ts, k_1sacc_cdn_ch_mean - k_1sacc_cdn_ch_ci, k_1sacc_cdn_ch_mean + k_1sacc_cdn_ch_ci, color=cdn_ch_color, alpha=alpha)
ax[1,0].plot(sacc_ts, k_1sacc_cdn_ch_mean, color = cdn_ch_color)


ax[1,0].fill_between(sacc_ts, k_1sacc_ofc_alt_unch_mean - k_1sacc_ofc_alt_unch_ci, k_1sacc_ofc_alt_unch_mean + k_1sacc_ofc_alt_unch_ci, color=ofc_alt_unch_color, alpha=alpha)
ax[1,0].plot(sacc_ts, k_1sacc_ofc_alt_unch_mean, color = ofc_alt_unch_color)

ax[1,0].fill_between(sacc_ts, k_1sacc_cdn_alt_unch_mean - k_1sacc_cdn_alt_unch_ci, k_1sacc_cdn_alt_unch_mean + k_1sacc_cdn_alt_unch_ci, color=cdn_alt_unch_color, alpha=alpha)
ax[1,0].plot(sacc_ts, k_1sacc_cdn_alt_unch_mean, color = cdn_alt_unch_color)

ax[1,0].set_ylim(subject_k_y_ylims)


# first saccade of double saccade trial
ax[1,1].fill_between(sacc_ts, k_1sacc2_ofc_ch_mean - k_1sacc2_ofc_ch_ci, k_1sacc2_ofc_ch_mean + k_1sacc2_ofc_ch_ci, color=ofc_ch_color, alpha=alpha)
ax[1,1].plot(sacc_ts, k_1sacc2_ofc_ch_mean, color = ofc_ch_color)

ax[1,1].fill_between(sacc_ts, k_1sacc2_cdn_ch_mean - k_1sacc2_cdn_ch_ci, k_1sacc2_cdn_ch_mean + k_1sacc2_cdn_ch_ci, color=cdn_ch_color, alpha=alpha)
ax[1,1].plot(sacc_ts, k_1sacc2_cdn_ch_mean, color = cdn_ch_color)

ax[1,1].fill_between(sacc_ts, k_1sacc2_ofc_alt_unch_mean - k_1sacc2_ofc_alt_unch_ci, k_1sacc2_ofc_alt_unch_mean + k_1sacc2_ofc_alt_unch_ci, color=ofc_alt_unch_color, alpha=alpha)
ax[1,1].plot(sacc_ts, k_1sacc2_ofc_alt_unch_mean, color = ofc_alt_unch_color)

ax[1,1].fill_between(sacc_ts, k_1sacc2_cdn_alt_unch_mean - k_1sacc2_cdn_alt_unch_ci, k_1sacc2_cdn_alt_unch_mean + k_1sacc2_cdn_alt_unch_ci, color=cdn_alt_unch_color, alpha=alpha)
ax[1,1].plot(sacc_ts, k_1sacc2_cdn_alt_unch_mean, color = cdn_alt_unch_color)

ax[1,1].set_ylim(subject_k_y_ylims)



# first saccade of double saccade trial
ax[1,2].fill_between(sacc_ts, k_2sacc2_ofc_ch_mean - k_2sacc2_ofc_ch_ci, k_2sacc2_ofc_ch_mean + k_2sacc2_ofc_ch_ci, color=ofc_ch_color, alpha=alpha)
ax[1,2].plot(sacc_ts, k_2sacc2_ofc_ch_mean, color = ofc_ch_color)

ax[1,2].fill_between(sacc_ts, k_2sacc2_cdn_ch_mean - k_2sacc2_cdn_ch_ci, k_2sacc2_cdn_ch_mean + k_2sacc2_cdn_ch_ci, color=cdn_ch_color, alpha=alpha)
ax[1,2].plot(sacc_ts, k_2sacc2_cdn_ch_mean, color = cdn_ch_color)

ax[1,2].fill_between(sacc_ts, k_2sacc2_ofc_alt_unch_mean - k_2sacc2_ofc_alt_unch_ci, k_2sacc2_ofc_alt_unch_mean + k_2sacc2_ofc_alt_unch_ci, color=ofc_alt_unch_color, alpha=alpha)
ax[1,2].plot(sacc_ts, k_2sacc2_ofc_alt_unch_mean, color = ofc_alt_unch_color)

ax[1,2].fill_between(sacc_ts, k_2sacc2_cdn_alt_unch_mean - k_2sacc2_cdn_alt_unch_ci, k_2sacc2_cdn_alt_unch_mean + k_2sacc2_cdn_alt_unch_ci, color=cdn_alt_unch_color, alpha=alpha)
ax[1,2].plot(sacc_ts, k_2sacc2_cdn_alt_unch_mean, color = cdn_alt_unch_color)

ax[1,2].set_ylim(subject_k_y_ylims)

# save the figure
#fig.savefig('saccade_aligned_latency_comparison.svg')

In [ ]:
# NOW ASSESSED SOME VERY PRECISELY MATCHED CONDITIONS

match_ix = (bhv['state'] < 3) & (unch_val == 1)

# create indices for the single and double saccade trials
single_ix = (bhv['n_sacc'] == 1)
double_ix = (bhv['n_sacc'] == 2)

# create indices for each subject
d_ix = subject == 0
k_ix = subject == 1

# get means and intervals

#----------------------------
# SUBJECT D
# OFC
# first saccade of single saccade trial
d_1sacc_ofc_ch_mean_matched, d_1sacc_ofc_ch_ci_matched = calculate_mean_and_interval(first_sacc_ofc_pps[single_ix & d_ix & match_ix,:, 0], 'bootstrap')
d_1sacc_ofc_alt_unch_mean_matched, d_1sacc_ofc_alt_unch_ci_matched = calculate_mean_and_interval(first_sacc_ofc_pps[single_ix & d_ix,:, 3], 'bootstrap')

# first saccade of double saccade trial
d_1sacc2_ofc_ch_mean_matched, d_1sacc2_ofc_ch_ci_matched = calculate_mean_and_interval(first_sacc_ofc_pps[double_ix & d_ix & match_ix,:, 0], 'bootstrap')
d_1sacc2_ofc_alt_unch_mean_matched, d_1sacc2_ofc_alt_unch_ci_matched = calculate_mean_and_interval(first_sacc_ofc_pps[double_ix & d_ix,:, 3], 'bootstrap')

# second saccade of double saccade trial
d_2sacc2_ofc_ch_mean_matched, d_2sacc2_ofc_ch_ci_matched = calculate_mean_and_interval(second_sacc_ofc_pps[double_ix & d_ix & match_ix,:, 0], 'bootstrap')
d_2sacc2_ofc_alt_unch_mean_matched, d_2sacc2_ofc_alt_unch_ci_matched = calculate_mean_and_interval(second_sacc_ofc_pps[double_ix & d_ix & match_ix,:, 3], 'bootstrap')

# CDN
# first saccade of single saccade trial
d_1sacc_cdn_ch_mean_matched, d_1sacc_cdn_ch_ci_matched = calculate_mean_and_interval(first_sacc_cdn_pps[single_ix & d_ix & match_ix,:, 0], 'bootstrap')
d_1sacc_cdn_alt_unch_mean_matched, d_1sacc_cdn_alt_unch_ci_matched = calculate_mean_and_interval(first_sacc_cdn_pps[single_ix & d_ix & match_ix,:, 3], 'bootstrap')

# first saccade of double saccade trial
d_1sacc2_cdn_ch_mean_matched, d_1sacc2_cdn_ch_ci_matched = calculate_mean_and_interval(first_sacc_cdn_pps[double_ix & d_ix & match_ix,:, 0], 'bootstrap')
d_1sacc2_cdn_alt_unch_mean_matched, d_1sacc2_cdn_alt_unch_ci_matched = calculate_mean_and_interval(first_sacc_cdn_pps[double_ix & d_ix & match_ix,:, 3], 'bootstrap')

# second saccade of double saccade trial
d_2sacc2_cdn_ch_mean_matched, d_2sacc2_cdn_ch_ci_matched = calculate_mean_and_interval(second_sacc_cdn_pps[double_ix & d_ix & match_ix,:, 0], 'bootstrap')
d_2sacc2_cdn_alt_unch_mean_matched, d_2sacc2_cdn_alt_unch_ci_matched = calculate_mean_and_interval(second_sacc_cdn_pps[double_ix & d_ix & match_ix,:, 3], 'bootstrap')


#----------------------------
# SUBJECT K
# OFC
# first saccade of single saccade trial
k_1sacc_ofc_ch_mean_matched, k_1sacc_ofc_ch_ci_matched = calculate_mean_and_interval(first_sacc_ofc_pps[single_ix & k_ix & match_ix,:, 0], 'bootstrap')
k_1sacc_ofc_alt_unch_mean_matched, k_1sacc_ofc_alt_unch_ci_matched = calculate_mean_and_interval(first_sacc_ofc_pps[single_ix & k_ix,:, 3], 'bootstrap')

# first saccade of double saccade trial
k_1sacc2_ofc_ch_mean_matched, k_1sacc2_ofc_ch_ci_matched = calculate_mean_and_interval(first_sacc_ofc_pps[double_ix & k_ix & match_ix,:, 0], 'bootstrap')
k_1sacc2_ofc_alt_unch_mean_matched, k_1sacc2_ofc_alt_unch_ci_matched = calculate_mean_and_interval(first_sacc_ofc_pps[double_ix & k_ix,:, 3], 'bootstrap')

# second saccade of double saccade trial
k_2sacc2_ofc_ch_mean_matched, k_2sacc2_ofc_ch_ci_matched = calculate_mean_and_interval(second_sacc_ofc_pps[double_ix & k_ix & match_ix,:, 0], 'bootstrap')
k_2sacc2_ofc_alt_unch_mean_matched, k_2sacc2_ofc_alt_unch_ci_matched = calculate_mean_and_interval(second_sacc_ofc_pps[double_ix & k_ix & match_ix,:, 3], 'bootstrap')

# CDN
# first saccade of single saccade trial
k_1sacc_cdn_ch_mean_matched, k_1sacc_cdn_ch_ci_matched = calculate_mean_and_interval(first_sacc_cdn_pps[single_ix & k_ix & match_ix,:, 0], 'bootstrap')
k_1sacc_cdn_alt_unch_mean_matched, k_1sacc_cdn_alt_unch_ci_matched = calculate_mean_and_interval(first_sacc_cdn_pps[single_ix & k_ix & match_ix,:, 3], 'bootstrap')

# first saccade of double saccade trial
k_1sacc2_cdn_ch_mean_matched, k_1sacc2_cdn_ch_ci_matched = calculate_mean_and_interval(first_sacc_cdn_pps[double_ix & k_ix & match_ix,:, 0], 'bootstrap')
k_1sacc2_cdn_alt_unch_mean_matched, k_1sacc2_cdn_alt_unch_ci_matched = calculate_mean_and_interval(first_sacc_cdn_pps[double_ix & k_ix & match_ix,:, 3], 'bootstrap')

# second saccade of double saccade trial
k_2sacc2_cdn_ch_mean_matched, k_2sacc2_cdn_ch_ci_matched = calculate_mean_and_interval(second_sacc_cdn_pps[double_ix & k_ix & match_ix,:, 0], 'bootstrap')
k_2sacc2_cdn_alt_unch_mean_matched, k_2sacc2_cdn_alt_unch_ci_matched = calculate_mean_and_interval(second_sacc_cdn_pps[double_ix & k_ix & match_ix,:, 3], 'bootstrap')


In [ ]:
# PLOT THE MATCHED CONDITIONS

fig, ax = plt.subplots(2, 3, figsize=(8, 5))
plt.subplots_adjust(wspace=0.4, hspace=0.4)  # add some white space

ofc_ch_color = 'tab:red'
cdn_ch_color = 'tab:orange'

ofc_alt_unch_color = 'tab:blue'
cdn_alt_unch_color = 'tab:purple'

subject_d_y_ylims = (0, .7)
subject_k_y_ylims = (0, .9)


# SUBJECT D

# first saccade of single saccade trial
ax[0,0].fill_between(sacc_ts, d_1sacc_ofc_ch_mean_matched - d_1sacc_ofc_ch_ci_matched, d_1sacc_ofc_ch_mean_matched + d_1sacc_ofc_ch_ci_matched, color=ofc_ch_color, alpha=alpha)
ax[0,0].plot(sacc_ts, d_1sacc_ofc_ch_mean_matched, color = ofc_ch_color)

ax[0,0].fill_between(sacc_ts, d_1sacc_cdn_ch_mean_matched - d_1sacc_cdn_ch_ci_matched, d_1sacc_cdn_ch_mean_matched + d_1sacc_cdn_ch_ci_matched, color=cdn_ch_color, alpha=alpha)
ax[0,0].plot(sacc_ts, d_1sacc_cdn_ch_mean_matched, color = cdn_ch_color)


ax[0,0].fill_between(sacc_ts, d_1sacc_ofc_alt_unch_mean_matched - d_1sacc_ofc_alt_unch_ci_matched, d_1sacc_ofc_alt_unch_mean_matched + d_1sacc_ofc_alt_unch_ci_matched, color=ofc_alt_unch_color, alpha=alpha)
ax[0,0].plot(sacc_ts, d_1sacc_ofc_alt_unch_mean_matched, color = ofc_alt_unch_color)

ax[0,0].fill_between(sacc_ts, d_1sacc_cdn_alt_unch_mean_matched - d_1sacc_cdn_alt_unch_ci_matched, d_1sacc_cdn_alt_unch_mean_matched + d_1sacc_cdn_alt_unch_ci_matched, color=cdn_alt_unch_color, alpha=alpha)
ax[0,0].plot(sacc_ts, d_1sacc_cdn_alt_unch_mean_matched, color = cdn_alt_unch_color)

ax[0,0].set_ylim(subject_d_y_ylims)


# first saccade of double saccade trial
ax[0,1].fill_between(sacc_ts, d_1sacc2_ofc_ch_mean_matched - d_1sacc2_ofc_ch_ci_matched, d_1sacc2_ofc_ch_mean_matched + d_1sacc2_ofc_ch_ci_matched, color=ofc_ch_color, alpha=alpha)
ax[0,1].plot(sacc_ts, d_1sacc2_ofc_ch_mean_matched, color = ofc_ch_color)

ax[0,1].fill_between(sacc_ts, d_1sacc2_cdn_ch_mean_matched - d_1sacc2_cdn_ch_ci_matched, d_1sacc2_cdn_ch_mean_matched + d_1sacc2_cdn_ch_ci_matched, color=cdn_ch_color, alpha=alpha)
ax[0,1].plot(sacc_ts, d_1sacc2_cdn_ch_mean_matched, color = cdn_ch_color)

ax[0,1].fill_between(sacc_ts, d_1sacc2_ofc_alt_unch_mean_matched - d_1sacc2_ofc_alt_unch_ci_matched, d_1sacc2_ofc_alt_unch_mean_matched + d_1sacc2_ofc_alt_unch_ci_matched, color=ofc_alt_unch_color, alpha=alpha)
ax[0,1].plot(sacc_ts, d_1sacc2_ofc_alt_unch_mean_matched, color = ofc_alt_unch_color)

ax[0,1].fill_between(sacc_ts, d_1sacc2_cdn_alt_unch_mean_matched - d_1sacc2_cdn_alt_unch_ci_matched, d_1sacc2_cdn_alt_unch_mean_matched + d_1sacc2_cdn_alt_unch_ci_matched, color=cdn_alt_unch_color, alpha=alpha)
ax[0,1].plot(sacc_ts, d_1sacc2_cdn_alt_unch_mean_matched, color = cdn_alt_unch_color)

ax[0,1].set_ylim(subject_d_y_ylims)



# first saccade of double saccade trial
ax[0,2].fill_between(sacc_ts, d_2sacc2_ofc_ch_mean_matched - d_2sacc2_ofc_ch_ci_matched, d_2sacc2_ofc_ch_mean_matched + d_2sacc2_ofc_ch_ci_matched, color=ofc_ch_color, alpha=alpha)
ax[0,2].plot(sacc_ts, d_2sacc2_ofc_ch_mean_matched, color = ofc_ch_color)

ax[0,2].fill_between(sacc_ts, d_2sacc2_cdn_ch_mean_matched - d_2sacc2_cdn_ch_ci_matched, d_2sacc2_cdn_ch_mean_matched + d_2sacc2_cdn_ch_ci_matched, color=cdn_ch_color, alpha=alpha)
ax[0,2].plot(sacc_ts, d_2sacc2_cdn_ch_mean_matched, color = cdn_ch_color)

ax[0,2].fill_between(sacc_ts, d_2sacc2_ofc_alt_unch_mean_matched - d_2sacc2_ofc_alt_unch_ci_matched, d_2sacc2_ofc_alt_unch_mean_matched + d_2sacc2_ofc_alt_unch_ci_matched, color=ofc_alt_unch_color, alpha=alpha)
ax[0,2].plot(sacc_ts, d_2sacc2_ofc_alt_unch_mean_matched, color = ofc_alt_unch_color)

ax[0,2].fill_between(sacc_ts, d_2sacc2_cdn_alt_unch_mean_matched - d_2sacc2_cdn_alt_unch_ci_matched, d_2sacc2_cdn_alt_unch_mean_matched + d_2sacc2_cdn_alt_unch_ci_matched, color=cdn_alt_unch_color, alpha=alpha)
ax[0,2].plot(sacc_ts, d_2sacc2_cdn_alt_unch_mean_matched, color = cdn_alt_unch_color)

ax[0,2].set_ylim(subject_d_y_ylims)


# SUBJECT K
# first saccade of single saccade trial
ax[1,0].fill_between(sacc_ts, k_1sacc_ofc_ch_mean_matched - k_1sacc_ofc_ch_ci_matched, k_1sacc_ofc_ch_mean_matched + k_1sacc_ofc_ch_ci_matched, color=ofc_ch_color, alpha=alpha)
ax[1,0].plot(sacc_ts, k_1sacc_ofc_ch_mean_matched, color = ofc_ch_color)

ax[1,0].fill_between(sacc_ts, k_1sacc_cdn_ch_mean_matched - k_1sacc_cdn_ch_ci_matched, k_1sacc_cdn_ch_mean_matched + k_1sacc_cdn_ch_ci_matched, color=cdn_ch_color, alpha=alpha)
ax[1,0].plot(sacc_ts, k_1sacc_cdn_ch_mean_matched, color = cdn_ch_color)


ax[1,0].fill_between(sacc_ts, k_1sacc_ofc_alt_unch_mean_matched - k_1sacc_ofc_alt_unch_ci_matched, k_1sacc_ofc_alt_unch_mean_matched + k_1sacc_ofc_alt_unch_ci_matched, color=ofc_alt_unch_color, alpha=alpha)
ax[1,0].plot(sacc_ts, k_1sacc_ofc_alt_unch_mean_matched, color = ofc_alt_unch_color)

ax[1,0].fill_between(sacc_ts, k_1sacc_cdn_alt_unch_mean_matched - k_1sacc_cdn_alt_unch_ci_matched, k_1sacc_cdn_alt_unch_mean_matched + k_1sacc_cdn_alt_unch_ci_matched, color=cdn_alt_unch_color, alpha=alpha)
ax[1,0].plot(sacc_ts, k_1sacc_cdn_alt_unch_mean_matched, color = cdn_alt_unch_color)

ax[1,0].set_ylim(subject_k_y_ylims)


# first saccade of double saccade trial
ax[1,1].fill_between(sacc_ts, k_1sacc2_ofc_ch_mean_matched - k_1sacc2_ofc_ch_ci_matched, k_1sacc2_ofc_ch_mean_matched + k_1sacc2_ofc_ch_ci_matched, color=ofc_ch_color, alpha=alpha)
ax[1,1].plot(sacc_ts, k_1sacc2_ofc_ch_mean_matched, color = ofc_ch_color)

ax[1,1].fill_between(sacc_ts, k_1sacc2_cdn_ch_mean_matched - k_1sacc2_cdn_ch_ci_matched, k_1sacc2_cdn_ch_mean_matched + k_1sacc2_cdn_ch_ci_matched, color=cdn_ch_color, alpha=alpha)
ax[1,1].plot(sacc_ts, k_1sacc2_cdn_ch_mean_matched, color = cdn_ch_color)

ax[1,1].fill_between(sacc_ts, k_1sacc2_ofc_alt_unch_mean_matched - k_1sacc2_ofc_alt_unch_ci_matched, k_1sacc2_ofc_alt_unch_mean_matched + k_1sacc2_ofc_alt_unch_ci_matched, color=ofc_alt_unch_color, alpha=alpha)
ax[1,1].plot(sacc_ts, k_1sacc2_ofc_alt_unch_mean_matched, color = ofc_alt_unch_color)

ax[1,1].fill_between(sacc_ts, k_1sacc2_cdn_alt_unch_mean_matched - k_1sacc2_cdn_alt_unch_ci_matched, k_1sacc2_cdn_alt_unch_mean_matched + k_1sacc2_cdn_alt_unch_ci_matched, color=cdn_alt_unch_color, alpha=alpha)
ax[1,1].plot(sacc_ts, k_1sacc2_cdn_alt_unch_mean_matched, color = cdn_alt_unch_color)

ax[1,1].set_ylim(subject_k_y_ylims)



# first saccade of double saccade trial
ax[1,2].fill_between(sacc_ts, k_2sacc2_ofc_ch_mean_matched - k_2sacc2_ofc_ch_ci_matched, k_2sacc2_ofc_ch_mean_matched + k_2sacc2_ofc_ch_ci_matched, color=ofc_ch_color, alpha=alpha)
ax[1,2].plot(sacc_ts, k_2sacc2_ofc_ch_mean_matched, color = ofc_ch_color)

ax[1,2].fill_between(sacc_ts, k_2sacc2_cdn_ch_mean_matched - k_2sacc2_cdn_ch_ci_matched, k_2sacc2_cdn_ch_mean_matched + k_2sacc2_cdn_ch_ci_matched, color=cdn_ch_color, alpha=alpha)
ax[1,2].plot(sacc_ts, k_2sacc2_cdn_ch_mean_matched, color = cdn_ch_color)

ax[1,2].fill_between(sacc_ts, k_2sacc2_ofc_alt_unch_mean_matched - k_2sacc2_ofc_alt_unch_ci_matched, k_2sacc2_ofc_alt_unch_mean_matched + k_2sacc2_ofc_alt_unch_ci_matched, color=ofc_alt_unch_color, alpha=alpha)
ax[1,2].plot(sacc_ts, k_2sacc2_ofc_alt_unch_mean_matched, color = ofc_alt_unch_color)

ax[1,2].fill_between(sacc_ts, k_2sacc2_cdn_alt_unch_mean_matched - k_2sacc2_cdn_alt_unch_ci_matched, k_2sacc2_cdn_alt_unch_mean_matched + k_2sacc2_cdn_alt_unch_ci_matched, color=cdn_alt_unch_color, alpha=alpha)
ax[1,2].plot(sacc_ts, k_2sacc2_cdn_alt_unch_mean_matched, color = cdn_alt_unch_color)

ax[1,2].set_ylim(subject_k_y_ylims)

# save the figure
#fig.savefig('saccade_aligned_matched_conds.svg')

In [ ]:
# let's make a new plot
post_sacc_mask = (sacc_ts >= 0) & (sacc_ts <= 200)

# --- SUBJECT K ---
# first saccade of single saccade trials
k_ofc_ch_sacc1_1      = np.nanmean(first_sacc_ofc_pps[single_ix & k_ix & match_ix][:, post_sacc_mask, 0], axis=1)
k_ofc_alt_unch_sacc1_1 = np.nanmean(first_sacc_ofc_pps[single_ix & k_ix & match_ix][:, post_sacc_mask, 3], axis=1)
k_cdn_ch_sacc1_1      = np.nanmean(first_sacc_cdn_pps[single_ix & k_ix & match_ix][:, post_sacc_mask, 0], axis=1)
k_cdn_alt_unch_sacc1_1 = np.nanmean(first_sacc_cdn_pps[single_ix & k_ix & match_ix][:, post_sacc_mask, 3], axis=1)

# first saccade of double saccade trials
k_ofc_ch_sacc2_1      = np.nanmean(first_sacc_ofc_pps[double_ix & k_ix & match_ix][:, post_sacc_mask, 0], axis=1)
k_ofc_alt_unch_sacc2_1 = np.nanmean(first_sacc_ofc_pps[double_ix & k_ix & match_ix][:, post_sacc_mask, 3], axis=1)
k_cdn_ch_sacc2_1      = np.nanmean(first_sacc_cdn_pps[double_ix & k_ix & match_ix][:, post_sacc_mask, 0], axis=1)
k_cdn_alt_unch_sacc2_1 = np.nanmean(first_sacc_cdn_pps[double_ix & k_ix & match_ix][:, post_sacc_mask, 3], axis=1)

# second saccade of double saccade trials
k_ofc_ch_sacc2_2      = np.nanmean(second_sacc_ofc_pps[double_ix & k_ix & match_ix][:, post_sacc_mask, 0], axis=1)
k_ofc_alt_unch_sacc2_2 = np.nanmean(second_sacc_ofc_pps[double_ix & k_ix & match_ix][:, post_sacc_mask, 3], axis=1)
k_cdn_ch_sacc2_2      = np.nanmean(second_sacc_cdn_pps[double_ix & k_ix & match_ix][:, post_sacc_mask, 0], axis=1)
k_cdn_alt_unch_sacc2_2 = np.nanmean(second_sacc_cdn_pps[double_ix & k_ix & match_ix][:, post_sacc_mask, 3], axis=1)

# --- SUBJECT D ---
# first saccade of single saccade trials
d_ofc_ch_sacc1_1      = np.nanmean(first_sacc_ofc_pps[single_ix & d_ix & match_ix][:, post_sacc_mask, 0], axis=1)
d_ofc_alt_unch_sacc1_1 = np.nanmean(first_sacc_ofc_pps[single_ix & d_ix & match_ix][:, post_sacc_mask, 3], axis=1)
d_cdn_ch_sacc1_1      = np.nanmean(first_sacc_cdn_pps[single_ix & d_ix & match_ix][:, post_sacc_mask, 0], axis=1)
d_cdn_alt_unch_sacc1_1 = np.nanmean(first_sacc_cdn_pps[single_ix & d_ix & match_ix][:, post_sacc_mask, 3], axis=1)

# first saccade of double saccade trials
d_ofc_ch_sacc2_1      = np.nanmean(first_sacc_ofc_pps[double_ix & d_ix & match_ix][:, post_sacc_mask, 0], axis=1)
d_ofc_alt_unch_sacc2_1 = np.nanmean(first_sacc_ofc_pps[double_ix & d_ix & match_ix][:, post_sacc_mask, 3], axis=1)
d_cdn_ch_sacc2_1      = np.nanmean(first_sacc_cdn_pps[double_ix & d_ix & match_ix][:, post_sacc_mask, 0], axis=1)
d_cdn_alt_unch_sacc2_1 = np.nanmean(first_sacc_cdn_pps[double_ix & d_ix & match_ix][:, post_sacc_mask, 3], axis=1)

# second saccade of double saccade trials
d_ofc_ch_sacc2_2      = np.nanmean(second_sacc_ofc_pps[double_ix & d_ix & match_ix][:, post_sacc_mask, 0], axis=1)
d_ofc_alt_unch_sacc2_2 = np.nanmean(second_sacc_ofc_pps[double_ix & d_ix & match_ix][:, post_sacc_mask, 3], axis=1)
d_cdn_ch_sacc2_2      = np.nanmean(second_sacc_cdn_pps[double_ix & d_ix & match_ix][:, post_sacc_mask, 0], axis=1)
d_cdn_alt_unch_sacc2_2 = np.nanmean(second_sacc_cdn_pps[double_ix & d_ix & match_ix][:, post_sacc_mask, 3], axis=1)

In [ ]:
for subject, ofc_ch_s1, ofc_ch_s2_1, ofc_ch_s2_2, ofc_unch_s1, ofc_unch_s2_1, ofc_unch_s2_2, \
             cdn_ch_s1, cdn_ch_s2_1, cdn_ch_s2_2, cdn_unch_s1, cdn_unch_s2_1, cdn_unch_s2_2 in zip(
    ['Subject D', 'Subject K'],
    [d_ofc_ch_sacc1_1,       k_ofc_ch_sacc1_1],
    [d_ofc_ch_sacc2_1,       k_ofc_ch_sacc2_1],
    [d_ofc_ch_sacc2_2,       k_ofc_ch_sacc2_2],
    [d_ofc_alt_unch_sacc1_1, k_ofc_alt_unch_sacc1_1],
    [d_ofc_alt_unch_sacc2_1, k_ofc_alt_unch_sacc2_1],
    [d_ofc_alt_unch_sacc2_2, k_ofc_alt_unch_sacc2_2],
    [d_cdn_ch_sacc1_1,       k_cdn_ch_sacc1_1],
    [d_cdn_ch_sacc2_1,       k_cdn_ch_sacc2_1],
    [d_cdn_ch_sacc2_2,       k_cdn_ch_sacc2_2],
    [d_cdn_alt_unch_sacc1_1, k_cdn_alt_unch_sacc1_1],
    [d_cdn_alt_unch_sacc2_1, k_cdn_alt_unch_sacc2_1],
    [d_cdn_alt_unch_sacc2_2, k_cdn_alt_unch_sacc2_2],
):
    print(f'\n{subject}')
    print('-' * 60)

    n_s1 = len(ofc_ch_s1)
    n_s2 = len(ofc_ch_s2_1)

    df_aov = pd.DataFrame({
        'posterior':    np.concatenate([
                            ofc_ch_s1,    ofc_ch_s2_1,    ofc_ch_s2_2,
                            ofc_unch_s1,  ofc_unch_s2_1,  ofc_unch_s2_2,
                            cdn_ch_s1,    cdn_ch_s2_1,    cdn_ch_s2_2,
                            cdn_unch_s1,  cdn_unch_s2_1,  cdn_unch_s2_2,
                        ]),
        'condition':    np.concatenate([
                            np.zeros(n_s1),  np.ones(n_s2),  np.full(n_s2, 2),
                            np.zeros(n_s1),  np.ones(n_s2),  np.full(n_s2, 2),
                            np.zeros(n_s1),  np.ones(n_s2),  np.full(n_s2, 2),
                            np.zeros(n_s1),  np.ones(n_s2),  np.full(n_s2, 2),
                        ]),
        'brain_area':   np.concatenate([
                            np.ones(n_s1 + n_s2*2),
                            np.ones(n_s1 + n_s2*2),
                           -np.ones(n_s1 + n_s2*2),
                           -np.ones(n_s1 + n_s2*2),
                        ]),
        'post_type':    np.concatenate([
                            np.ones(n_s1 + n_s2*2),
                           -np.ones(n_s1 + n_s2*2),
                            np.ones(n_s1 + n_s2*2),
                           -np.ones(n_s1 + n_s2*2),
                        ]),
        'trial':        np.concatenate([
                            np.arange(n_s1),           np.arange(n_s2) + 10000,  np.arange(n_s2) + 20000,
                            np.arange(n_s1) + 30000,   np.arange(n_s2) + 40000,  np.arange(n_s2) + 50000,
                            np.arange(n_s1) + 60000,   np.arange(n_s2) + 70000,  np.arange(n_s2) + 80000,
                            np.arange(n_s1) + 90000,   np.arange(n_s2) + 100000, np.arange(n_s2) + 110000,
                        ])
    })

    df_aov['condition']  = df_aov['condition'].astype('category')
    df_aov['brain_area'] = df_aov['brain_area'].astype('category')
    df_aov['post_type']  = df_aov['post_type'].astype('category')

    aov = pg.anova(data=df_aov, dv='posterior',
                   between=['condition', 'brain_area', 'post_type'],
                   detailed=True)
    print(aov[['Source', 'DF', 'MS', 'F', 'p-unc', 'np2']].to_string())

In [ ]:
for subject, cdn_s1, cdn_s2_1, cdn_s2_2 in zip(
    ['Subject D', 'Subject K'],
    [d_cdn_ch_sacc1_1, k_cdn_ch_sacc1_1],
    [d_cdn_ch_sacc2_1, k_cdn_ch_sacc2_1],
    [d_cdn_ch_sacc2_2, k_cdn_ch_sacc2_2],
):
    print(f'\n{subject}')
    print('-' * 60)

    n_s1 = len(cdn_s1)
    n_s2 = len(cdn_s2_1)

    df_aov = pd.DataFrame({
        'posterior': np.concatenate([cdn_s1, cdn_s2_1, cdn_s2_2]),
        'condition': np.concatenate([
                        np.zeros(n_s1),
                        np.ones(n_s2),
                        np.full(n_s2, 2)
                     ]),
    })
    df_aov['condition'] = df_aov['condition'].astype('category')

    # one-way ANOVA
    aov = pg.anova(data=df_aov, dv='posterior', between='condition', detailed=True)
    print('One-way ANOVA:')
    print(aov[['Source', 'DF', 'F', 'p-unc', 'np2']].to_string())

    # marginal means
    print('\nMarginal means:')
    print(df_aov.groupby('condition')['posterior'].mean().to_string())

    # pairwise comparisons
    print('\nPairwise comparisons (FDR corrected):')
    pwc = pg.pairwise_tests(data=df_aov, dv='posterior', between='condition',
                            padjust='fdr_bh')
    print(pwc[['A', 'B', 'T', 'dof', 'p-unc', 'p-corr', 'hedges']].to_string())

In [ ]:
for subject, ofc_s1, ofc_s2_1, ofc_s2_2, cdn_s1, cdn_s2_1, cdn_s2_2 in zip(
    ['Subject D', 'Subject K'],
    [d_ofc_ch_sacc1_1, k_ofc_ch_sacc1_1],
    [d_ofc_ch_sacc2_1, k_ofc_ch_sacc2_1],
    [d_ofc_ch_sacc2_2, k_ofc_ch_sacc2_2],
    [d_cdn_ch_sacc1_1, k_cdn_ch_sacc1_1],
    [d_cdn_ch_sacc2_1, k_cdn_ch_sacc2_1],
    [d_cdn_ch_sacc2_2, k_cdn_ch_sacc2_2],
):
    print(f'\n{subject}')
    for area_label, s1, s2_1, s2_2 in zip(
        ['OFC', 'CdN'],
        [ofc_s1,  cdn_s1],
        [ofc_s2_1, cdn_s2_1],
        [ofc_s2_2, cdn_s2_2]
    ):
        n_s1 = len(s1)
        n_s2 = len(s2_1)

        df_pwc = pd.DataFrame({
            'posterior': np.concatenate([s1, s2_1, s2_2]),
            'condition': np.concatenate([
                            np.zeros(n_s1),
                            np.ones(n_s2),
                            np.full(n_s2, 2)
                         ]),
        })
        df_pwc['condition'] = df_pwc['condition'].astype('category')

        pwc = pg.pairwise_tests(data=df_pwc, dv='posterior', between='condition',
                                padjust='fdr_bh')
        print(f'\n{area_label}:')
        print(pwc[['A', 'B', 'T', 'dof', 'p-unc', 'p-corr', 'hedges']].to_string())

In [ ]:
# compare pics vs saccade-aligned PPs

# get pps during the first 300 milliseconds from pics on
ofc_pics_pp = np.nanmean(OFC_ch[:, 41:54], axis=1)
cdn_pics_pp = np.nanmean(CdN_ch[:, 41:54], axis=1)

# get pps during the 300 ms from a saccade
ofc_sacc_pp = np.nanmean(first_sacc_ofc_pps[:, 15:,0], axis=1)
cdn_sacc_pp = np.nanmean(first_sacc_cdn_pps[:, 15:,0], axis=1)

k_ofc_means = np.zeros(shape=(4,2))
k_ofc_cis = np.zeros(shape=(4,2))
k_cdn_means = np.zeros(shape=(4,2))
k_cdn_cis = np.zeros(shape=(4,2))

d_ofc_means = np.zeros(shape=(4,2))
d_ofc_cis = np.zeros(shape=(4,2))
d_cdn_means = np.zeros(shape=(4,2))
d_cdn_cis = np.zeros(shape=(4,2))

for s_ix, this_session in enumerate(bhv['fname'].loc[k_ix].unique()):

    this_session_ix = bhv['fname'] == this_session

    k_ofc_means[s_ix,0], k_ofc_cis[s_ix,0] = calculate_mean_and_interval(ofc_pics_pp[this_session_ix], 'bootstrap')
    k_ofc_means[s_ix,1], k_ofc_cis[s_ix,1] = calculate_mean_and_interval(ofc_sacc_pp[this_session_ix], 'bootstrap')
    k_cdn_means[s_ix,0], k_cdn_cis[s_ix,0] = calculate_mean_and_interval(cdn_pics_pp[this_session_ix], 'bootstrap')
    k_cdn_means[s_ix,1], k_cdn_cis[s_ix,1] = calculate_mean_and_interval(cdn_sacc_pp[this_session_ix], 'bootstrap')

for s_ix, this_session in enumerate(bhv['fname'].loc[d_ix].unique()):

    this_session_ix = bhv['fname'] == this_session

    d_ofc_means[s_ix,0], d_ofc_cis[s_ix,0] = calculate_mean_and_interval(ofc_pics_pp[this_session_ix], 'bootstrap')
    d_ofc_means[s_ix,1], d_ofc_cis[s_ix,1] = calculate_mean_and_interval(ofc_sacc_pp[this_session_ix], 'bootstrap')
    d_cdn_means[s_ix,0], d_cdn_cis[s_ix,0] = calculate_mean_and_interval(cdn_pics_pp[this_session_ix], 'bootstrap')
    d_cdn_means[s_ix,1], d_cdn_cis[s_ix,1] = calculate_mean_and_interval(cdn_sacc_pp[this_session_ix], 'bootstrap')


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(8, 4))

ax[0].plot(k_ofc_means[:, 0], k_ofc_means[:, 1], 'o', color='tab:blue', markersize=6)
ax[0].plot(k_cdn_means[:, 0], k_cdn_means[:, 1], 'o', color='tab:red', markersize=6)
ax[0].plot([0,1], [0,1], 'k', lw = .5)
ax[0].set_xlim((0,1))
ax[0].set_ylim((0,1))

ax[1].plot(d_ofc_means[:, 0], d_ofc_means[:, 1], 'o', color='tab:blue', markersize=6)
ax[1].plot(d_cdn_means[:, 0], d_cdn_means[:, 1], 'o', color='tab:red', markersize=6)
ax[1].plot([0,1], [0,1], 'k', lw = .5)
ax[1].set_xlim((0,1))
ax[1].set_ylim((0,1))

#fig.savefig('comparing_decoders.svg')

In [ ]:
pg.ttest(d_ofc_means[:, 0], d_ofc_means[:, 1], paired=True)